# Spark SQL Learning 
##  By knowing the process of Read and Write , become a  Data INGESTION Developer 

connecting various sources (files or filesustem , db , dwh , api ,.. ) and loading data into storage env(data lake)

- csv
- json
- xml
- parquet 
- ORC
- sql
etc....


## Few Facts about Unity Catalog

Dataobjects  is managed  in catalog with three namespace 


Catalog -> per domain or environemnt 

schema -> database 

tables , views , functions , volume 

Volume -  Non tabular data (files) , goverened access 

Catalog >> Schema  >> 
                    Table
                    View
                    functions
                    volume 

Volumes are used for managing Non tabulat data (files)

In [0]:
%sql
create catalog if not exists izwd37dev;

create schema if  not exists izwd37dev.wd37db;

create volume if not exists izwd37dev.wd37db.rawdatta;



## DBFS 

### Databricks File system 

distribuited virtual file system , linux posix format  runninng on top of your cloud storages 


/Volumes/catalog/schema/volume-name/path/to/file



dbfs:/ - uri   -> uniform resource identifier  (dbricks file system )

hdfs:/    -> hadoop distruibuited file system 
file:/     -> local file 

s3a:/   -> aws s3

gcs:/  -> google storage


adls:/   -> azure datalake 





In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/"

In [0]:
print(dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta"))

In [0]:
%fs ls "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"

In [0]:
%fs head "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs"

In [0]:
%fs mkdirs "dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv"

In [0]:
dbutils.fs.ls("dbfs:/Volumes/izwd37dev/wd37db/rawdatta")

In [0]:
dbutils.fs.mkdirs("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json")

what is dbfs:/ -> uri 
        hdfs:/ 
        file:/
        s3:/
        gcs:/

In [0]:
# read data from spark 
# spark SQL 

Spark session 


from pyspark.sql import SparkSession 

spark=SparkSession.builder.getOrCreate()

In [0]:
print(spark)

create Dataframe from storage (files / dir )
To read the delimited data from any storage (dbfs , hdfs , lfs , cloud staorgaes )

spark.read.csv opition  -> dataframe 

-- csv is the built in source 


## Deafults
spark.read.csv 

default options :

header = False 

default cols = _c0 , _c1 _c2

delimiter = ","

default data type for all columns   = string 

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs") # file path
print(type(custdata)) # dataframe
custdata.show() # similar to collect -> action )
# show action , display default 20 records 

In [0]:
# describe table 
custdata.printSchema()

In [0]:
# view few or more records 
custdata.show(10,False)
#custdata.show(200)

In [0]:
display(custdata)

In [0]:
# changing the column names from _c0, _c1 to actuals 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs").toDF("custid","fname","lname","age","profession")

custdata.show(5) # display only 5 records 

custdata.printSchema()

### supose we recivied a file with header 
### column names we need to pick from the header 

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

custdata.show(5)

custdata.printSchema()

In [0]:
# how to get the number of records in the dataframe
# select count(1) from table 
# count is an action , its return integer result , trigger execution , job is created
custdata.count()

overriding the defaults with option 

enable the header 

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True)

custdata.show(5)

custdata.printSchema()
print(f"Number of records in cust : {custdata.count()}")

In [0]:
# data type for all columns treated default as string
# based on the data we have generate the schema with proper data type 
# performance if we read large data inferSchema is not a good option 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custs_header",header=True,inferSchema=True)

custdata.show(5)

custdata.printSchema()


In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp1.csv

In [0]:
emp_df=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,inferSchema=True)
emp_df.show()

emp_df.printSchema()


In [0]:
# options , way to create dataframe using csv with different options 
custdata=spark.read.csv(path="dbfs:/Volumes/izwd37dev/wd37db/rawdatta/csv") # directory 
custdata.show(5)
custdata.count()

In [0]:
# creating a datfrme from list of paths 
custdata=spark.read.csv(path=["/Volumes/izwd37dev/wd37db/rawdatta/csv/custs","/Volumes/izwd37dev/wd37db/rawdatta/csv/cust2.csv"]) # directory 
custdata.show(5)
custdata.count()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales_chennai.csv

In [0]:
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales*",header=True,inferSchema=True)
sales_df.show(1000)
print(sales_df.count())
sales_df.printSchema()

In [0]:
# recursiveFileLookup - read all sub dir
# pathGlobFilter - apply the filter / pattern globally on all dir /sub dir 
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata",header=True,recursiveFileLookup=True,pathGlobFilter="sales*")
sales_df.show(5)
print(sales_df.count())
sales_df.printSchema()